In [15]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Attention
from tensorflow.keras.layers import Dropout

In [16]:
X = np.load("../processed/X.npy")
y = np.load("../processed/y.npy")

print(X.shape)
print(y.shape)

(4635, 7, 297)
(4635,)


In [17]:
print(X.shape)

print(np.min(X))
print(np.max(X))
print(np.mean(X))
print(np.std(X))

(4635, 7, 297)
-5.29554426026541
71.4000913042482
3.6604406019981197
11.48431919646437


In [18]:
print(X.dtype)

float64


In [19]:
print(y[:50])

[1 0 0 1 1 0 1 0 1 0 0 1 0 1 0 0 1 0 1 0 1 0 1 1 0 0 1 1 0 0 1 0 1 0 0 1 1
 0 0 1 1 0 1 0 1 0 1 0 1 0]


In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_rf = X.reshape(X.shape[0], -1)

print(X_rf.shape)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train.reshape(X_train.shape[0], -1), y_train)

pred = rf.predict(
    X_test.reshape(X_test.shape[0], -1)
)

print(
    accuracy_score(
        y_test,
        pred
    )
)

(4635, 2079)
0.7454153182308522


In [27]:
from sklearn.preprocessing import StandardScaler

In [28]:
X_flat = X.reshape(-1, 297)

print(X_flat.shape)

(32445, 297)


In [29]:
scaler = StandardScaler()

X_flat_scaled = scaler.fit_transform(X_flat)

In [30]:
X_scaled = X_flat_scaled.reshape(
    X.shape[0],
    X.shape[1],
    X.shape[2]
)

print(X_scaled.shape)

(4635, 7, 297)


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(3708, 7, 297)
(927, 7, 297)


In [32]:
inputs = Input(shape=(7,297))

x = LSTM(
    64,
    return_sequences=True
)(inputs)

x = Dropout(0.3)(x)

x = LSTM(
    32,
    return_sequences=False
)(x)

x = Dropout(0.3)(x)

outputs = Dense(
    1,
    activation="sigmoid"
)(x)

model = Model(
    inputs,
    outputs
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 7, 297)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 7, 64)          │        92,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 7, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105,121 (410.63 KB)

 Trainable params: 105,121 (410.63 KB)

 Non-trainable params: 0 (0.00 B)

In [33]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [34]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=32
)

Epoch 1/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.5199 - loss: 0.6949 - val_accuracy: 0.5795 - val_loss: 0.6700
Epoch 2/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6431 - loss: 0.6326 - val_accuracy: 0.6173 - val_loss: 0.6528
Epoch 3/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7279 - loss: 0.5353 - val_accuracy: 0.6361 - val_loss: 0.6338
Epoch 4/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8166 - loss: 0.4101 - val_accuracy: 0.6577 - val_loss: 0.7194
Epoch 5/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8741 - loss: 0.2967 - val_accuracy: 0.6361 - val_loss: 0.8352
Epoch 6/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9131 - loss: 0.2189 - val_accuracy: 0.6361 - val_loss: 0.9943
Epoch 7/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9395 - loss: 0.1527 - val_accuracy: 0.6307 - val_loss: 1.1178
Epoch 8/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9583 - loss: 0.1124 - val_accuracy: 0

In [35]:
pred_prob = model.predict(X_test)

pred = (pred_prob > 0.5).astype(int)

print(
    classification_report(
        y_test,
        pred
    )
)

29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
              precision    recall  f1-score   support

           0       0.70      0.55      0.62       467
           1       0.63      0.76      0.69       460

    accuracy                           0.66       927
   macro avg       0.66      0.66      0.65       927
weighted avg       0.67      0.66      0.65       927



In [36]:
from tensorflow.keras.callbacks import EarlyStopping

In [37]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [38]:
inputs = Input(shape=(7,297))

In [39]:
x = LSTM(
    64,
    return_sequences=True,
    dropout=0.3,
    recurrent_dropout=0.3
)(inputs)

In [40]:
x = LSTM(
    32,
    return_sequences=False,
    dropout=0.3,
    recurrent_dropout=0.3
)(x)

In [41]:
outputs = Dense(
    1,
    activation="sigmoid"
)(x)

In [44]:
model = Model(
    inputs,
    outputs
)

In [45]:
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 7, 297)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 7, 64)          │        92,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105,121 (410.63 KB)

 Trainable params: 105,121 (410.63 KB)

 Non-trainable params: 0 (0.00 B)

In [46]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [47]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - accuracy: 0.5442 - loss: 0.6908 - val_accuracy: 0.5660 - val_loss: 0.6764
Epoch 2/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6122 - loss: 0.6610 - val_accuracy: 0.6199 - val_loss: 0.6574
Epoch 3/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.6470 - loss: 0.6233 - val_accuracy: 0.6496 - val_loss: 0.6209
Epoch 4/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.7003 - loss: 0.5750 - val_accuracy: 0.6307 - val_loss: 0.6345
Epoch 5/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7243 - loss: 0.5421 - val_accuracy: 0.6388 - val_loss: 0.6175
Epoch 6/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7399 - loss: 0.5049 - val_accuracy: 0.6253 - val_loss: 0.6355
Epoch 7/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.7830 - loss: 0.4619 - val_accuracy: 0.6388 - val_loss: 0.6547
Epoch 8/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.7923 - loss: 0.4292 - val_acc

In [48]:
pred_prob = model.predict(X_test)

29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step


In [49]:
pred = (pred_prob > 0.5).astype(int)

In [50]:
print(
    classification_report(
        y_test,
        pred
    )
)

              precision    recall  f1-score   support

           0       0.73      0.54      0.62       467
           1       0.63      0.80      0.70       460

    accuracy                           0.67       927
   macro avg       0.68      0.67      0.66       927
weighted avg       0.68      0.67      0.66       927



In [51]:
print(
    confusion_matrix(
        y_test,
        pred
    )
)

[[252 215]
 [ 94 366]]
